# Step 2 — Calibrate the tank turbulence (SettlingCalibration v01)

Calibrates the time-dependent diffusion coefficient $K(t)$ of the tank for **one mixer rotation rate** using a **size standard** of known size range and density, and writes a **calibration file** that `3_SettlingFit_v01.ipynb` (step 3) reads. Do this once per rotation rate (and again if the tank, water depth or sensor position changes).

**Inputs**
- the standard's OBS decay record (`sec, NTU, Co_NTU_avg, Cbackground_NTU_avg`, the `*_time_series_decayONLY.csv` file from step 1),
- the ADV decay record of the same mixer setting (from `0_ADV_data_tank_v01.ipynb`), which supplies the *shape* of the turbulence decay (optional: set `adv_file = None` if there is no ADV record),
- the standard's size range and density.

**Models:** both turbulence models are calibrated: $K(t)=K_{min}+K_0\,\mathrm{shape}(t)$ from the ADV (`'adv'`) and $K(t)=K_{min}+K_0e^{-t/\tau}$ (`'exp'`). Without an ADV record only `'exp'` is calibrated. The settling velocities of the standard are fixed by its size range and density; only the turbulence parameters ($K_0$, $K_{min}$, and $\tau$ for `'exp'`) are fitted to the decay. Step 3 can use either model.

**Output:** `calibration_file` (json), with the parameters of each calibrated model (and the ADV shape), the default model for step 3, rotation rate, tank geometry, fit quality, and how $K_0$ changes with the assumed density of the standard.

In [ ]:
# --- Find the project folder, then work inside its work/ folder ----------------
# All data paths below (data_OBS/, calibrations/, results/, ...) are relative to work/.
import os
import sys
_root = os.getcwd()
for _ in range(4):   # this folder, then up to three levels above
    if os.path.isdir(os.path.join(_root, 'tools')) and os.path.isdir(os.path.join(_root, 'work_template')):
        break
    _root = os.path.dirname(_root)
else:
    raise FileNotFoundError('could not find the project folder (the one containing tools/ and work_template/)')
sys.path.insert(0, _root)                          # for "from tools ..." imports
sys.path.insert(0, os.path.join(_root, 'tools'))   # for "import settling_tools"
if not os.path.isdir(os.path.join(_root, 'work')):
    from setup_work import setup_work
    setup_work(_root)                              # first run: create work/ from work_template/
os.chdir(os.path.join(_root, 'work'))
print('working in:', os.getcwd())
# ---------------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import settling_tools as ss

%config InlineBackend.figure_format='retina'

## 1. User input

In [ ]:
# --- User input -----------------------------------------------------------
# files
standard_obs_file = 'data_OBS/MicroSphere_75_90_SAMPLE_time_series_decayONLY.csv'   # OBS decay of the standard
adv_file          = 'data_ADV/filtered/ADV_02_60rpm_SAMPLE_filtered_decay_only.csv'   # ADV decay, same mixer rotation rate; None if there is none
calibration_file  = 'calibrations/cal_60rpm.json'                                       # written by this notebook

rpm   = 60  # mixer rotation rate of these runs (stored in the file, checked in step 3)
notes = ''  # free text stored in the file (tank, date, operator, ...)

# tank geometry (as measured, inches); the sensor height is measured UP from the bottom
water_depth_in         = 7.0
sensor_above_bottom_in = 2.5

# water and settling law
T_C   = 19.4 # deg C
S_ppt = 0.0  # salinity [ppt]
coef  = 5    # 1 flocs, 2 viscous flocs, 3 sieve sand, 4 natural sand, 5 smooth sphere (size standards)

# the size standard (a size RANGE of known density)
d_lo_um, d_hi_um, n_classes = 75.0, 90.0, 7
rho_standard = 1200.0  # [kg/m3]
psd_weights  = None   # None = uniform in diameter; or n_classes weights from a measured PSD

# turbulence models: both are calibrated ('exp' only when adv_file is None)
#   'adv': K(t) = Kmin + K0*shape(t), shape from the ADV;   'exp': K(t) = Kmin + K0*exp(-t/tau)
default_model = 'adv'   # model step 3 uses unless told otherwise; becomes 'exp' when adv_file is None
free_adv = ('K0', 'Kmin')          # parameters fitted for 'adv'
free_exp = ('K0', 'tau', 'Kmin')   # parameters fitted for 'exp' (one left out stays at its starting value)
K0_start, tau_start, Kmin_start = 1e-4, 15.0, 5e-6     # starting values [m2/s, s, m2/s]

# ADV shape settings (see settling_tools.adv_shape)
adv_window_s     = 2.0      # length of each averaging window [s]
adv_noise        = None     # noise floor [m2/s2]; None = estimate it from the late record
adv_noise_from_s = 60.0     # estimate the noise floor from the record after this time [s]
adv_cut_s        = 30.0     # set the shape to zero after this time [s]; Kmin carries the weak mixing beyond it
adv_normalize    = 'first'  # 'first': shape = 1 in the first window; 'max': shape = 1 at its maximum

# how K0 depends on the assumed density of the standard (kg/m3 offsets); () to skip (saves about a minute)
density_offsets = (-50, -25, 0, 25, 50)

# optional: test the calibration on a SECOND standard (different size range or density) without refitting
validate = False
val_obs_file = 'data_OBS/second_standard_time_series_decayONLY.csv'
val_d_lo_um, val_d_hi_um, val_rho = 150.0, 180.0, 2650.0
# --------------------------------------------------------------------------

geom = ss.Geometry(h=water_depth_in*0.0254, z_obs=sensor_above_bottom_in*0.0254)
print(f'h = {geom.h:.4f} m, sensor {geom.z_obs:.4f} m above the bottom, {geom.z_s:.4f} m below the surface')
print(f'water density {ss.density(T_C, S_ppt):.1f} kg/m3, viscosity {ss.visc(T_C):.3e} m2/s')

## 2. Plot the standard decay record

In [ ]:
# Load the standard's decay record; c is the normalised concentration (NTU - Cbackground) / (Co - Cbackground)
d = ss.load_decay(standard_obs_file)
t, c = d['t'], d['c']

dt = np.median(np.diff(t))   # sampling interval [s]
print(f"{len(t)} samples, dt = {dt:.1f} s, t = {t[0]:.0f} ... {t[-1]:.0f} s, "
      f"Co = {d['Co']:.3f} NTU, Cbackground = {d['Cb']:.3f} NTU")

# Plot the decay; the lines at 0 and 1 mark the background and initial concentrations
plt.figure(figsize=(6, 3.6))
plt.plot(t, c, '.', ms=3, color='0.4')
plt.axhline(0, color='k', lw=0.6)
plt.axhline(1, color='k', lw=0.6)
plt.xlabel('Time since mixer off [s]')
plt.ylabel('$(NTU-C_{bg})/(C_o-C_{bg})$')
plt.tight_layout()

## 3. Extract the turbulence decay shape from the ADV

Skipped when `adv_file = None`; then only the exponential model is calibrated.

Windows of a few seconds; in each the mean flow (linear trend) is removed and the fluctuation variance $\overline{v_1'^2+v_2'^2+v_3'^2}$ is computed. The noise floor (median of the late record) is subtracted and $K\propto\sqrt{E-E_{noise}}$. Only the **shape** is taken from the ADV; the amplitude $K_0$ comes from the fit below. The shape is set to zero where the ADV signal is indistinguishable from noise, and the persistent weak mixing is carried by $K_{min}$.

In [ ]:
if adv_file:
    # Dimensionless decay shape of the turbulence, from the ADV record
    shape = ss.adv_shape(adv_file, window_s=adv_window_s, noise=adv_noise, noise_from_s=adv_noise_from_s,
                         t_cut_s=adv_cut_s, normalize=adv_normalize)
    print(f"noise floor of the detrended fluctuation variance: {shape['noise']:.2e} m2/s2")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

    # Left: fluctuation variance in each window, with the noise floor that is subtracted
    ax[0].semilogy(shape['t'], shape['fluct'], 'o-', ms=3, lw=0.8)
    ax[0].axhline(shape['noise'], color='C3', ls=':', label='noise floor')
    ax[0].set_xlim(0, 60)
    ax[0].set_xlabel('Time since mixer off [s]')
    ax[0].set_ylabel('fluctuation variance [m$^2$/s$^2$]')
    ax[0].legend()

    # Right: the resulting shape, K(t)/K0
    ax[1].plot(shape['t'], shape['shape'], 'o-', ms=3)
    ax[1].set_xlim(0, 60)
    ax[1].set_xlabel('Time since mixer off [s]')
    ax[1].set_ylabel('K(t)/K0  (shape)')

    plt.tight_layout()
else:
    # No ADV record: only the exponential model is calibrated
    shape = None
    print('adv_file = None: no ADV shape; only the exponential model is calibrated')

## 4. Calibrate the turbulence models on the standard

Each model's parameters are fitted with the class velocities of the standard held fixed. The plot compares the models with the data; the model with the lower rms fits better, but similar rms values mean the OBS decay alone cannot tell them apart.

In [ ]:
# --- Size classes of the standard -------------------------------------------
# Split the nominal size range (d_lo_um to d_hi_um) into n_classes equal-width bins.
#   d_cls: bin-centre diameters [um]
#   p_cls: mass fraction in each bin (sums to 1; uniform unless psd_weights is given)
d_cls, p_cls = ss.size_classes(d_lo_um, d_hi_um, n_classes, psd_weights)

# Settling velocity of each class [m/s], from the known density of the standard
ws_cls = ss.class_ws(d_cls, None, T_C, S_ppt, coef, rhos=rho_standard)
print('class velocities [mm/s]:', np.round(ws_cls * 1e3, 3))

# --- Fit the turbulence parameters of each model ------------------------------
# With the class velocities held fixed, fit K(t) so the model matches the standard's decay.
# cals['exp'] is always made; cals['adv'] only when there is an ADV shape.
cals = ss.calibrate_models(t, c, geom, ws_cls, p_cls, shape=shape,
                           free=dict(adv=free_adv, exp=free_exp),
                           K0=K0_start, tau=tau_start, Kmin=Kmin_start)
if default_model not in cals:
    print(f"default_model = '{default_model}' was not calibrated; using 'exp'")
    default_model = 'exp'

# --- Report the fits ----------------------------------------------------------
for model, cal in cals.items():
    if model == 'exp':
        print(f"exp: K0 = {cal['K0']:.3g} m2/s, Kmin = {cal['Kmin']:.3g} m2/s, tau = {cal['tau']:.3g} s")
    else:
        print(f"adv: K0 = {cal['K0']:.3g} m2/s, Kmin = {cal['Kmin']:.3g} m2/s (ADV shape)")
    print(f"     rms = {cal['rms']:.4f}, lag-1 residual autocorrelation = {cal['rho1']:.2f}")
    # fractional 1-sigma uncertainty of each fitted parameter, inflated for autocorrelated residuals
    sigma_frac = {k: round(float(v), 3) for k, v in cal['sigma_ln'].items()}
    print('     1-sigma (fractional, autocorrelation-inflated):', sigma_frac)

# --- Plot: data vs each model (top) and residuals (bottom) ----------------------
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
ax[0].plot(t, c, 'k.', ms=3, label='standard (data)')
colors = {'adv': 'C3', 'exp': 'C0'}
for model, cal in cals.items():
    cm = ss.model_c_cal(t, ws_cls, p_cls, cal, geom, shape=shape)
    ax[0].plot(t, cm, '-', color=colors[model], label=f"{model} model (rms {cal['rms']:.4f})")
    ax[1].plot(t, c - cm, '-', color=colors[model], lw=0.8)
ax[0].set_title(f'standard {d_lo_um:.0f}-{d_hi_um:.0f} um at {rho_standard:.0f} kg/m3')
ax[0].set_ylabel('c/c0')
ax[0].legend()
ax[1].axhline(0, color='gray')
ax[1].set_xlabel('t since mixer off [s]')
ax[1].set_ylabel('residual')
plt.tight_layout()

## 5. How much does the assumed density of the standard matter?

$K_0$ and the standard's density trade off: a denser standard settles faster and needs more mixing to reproduce the same decay. The tables recalibrate each model for each density offset; the spread is the realistic uncertainty of $K_0$ and is stored in the calibration file.

In [ ]:
sens = {}   # one table per model
if density_offsets:
    # Densities to try: the nominal density of the standard plus each offset [kg/m3]
    rho_list = [rho_standard + offset for offset in density_offsets]

    for model, cal in cals.items():
        # Recalibrate K(t) at each density, starting from the nominal fit
        sens[model] = ss.calibration_sensitivity(t, c, geom, d_lo_um, d_hi_um, n_classes, rho_list, T_C, S_ppt, coef,
                                                 shape=shape if model == 'adv' else None, free=cal['free'],
                                                 K0=cal['K0'], tau=cal['tau'], Kmin=cal['Kmin'], weights=psd_weights)

        # K0 relative to the nominal calibration, as the second column
        sens[model].insert(1, 'K0 / K0(nominal)', sens[model]['K0'] / cal['K0'])

        # Show K0 and Kmin in scientific notation, everything else to 4 significant figures
        col_formats = {}
        for col in sens[model].columns:
            col_formats[col] = '{:.2e}' if col in ('K0', 'Kmin') else '{:.4g}'
        print(f'{model} model:')
        display(sens[model].style.format(col_formats).hide(axis='index'))

## 6. Save the calibration file

In [ ]:
# Description of the size standard, stored in the file for reference
if psd_weights is None:
    psd_weights_list = None
else:
    psd_weights_list = list(np.asarray(psd_weights, float))

standard = dict(d_lo_um=d_lo_um, d_hi_um=d_hi_um, n_classes=n_classes, rho_kg_m3=rho_standard,
                T_C=T_C, S_ppt=S_ppt, coef=coef, psd_weights=psd_weights_list)

# Names of the input files (without folders), stored for reference
files = dict(standard_obs_file=os.path.basename(standard_obs_file),
             adv_file=os.path.basename(adv_file) if adv_file else None)

ss.save_calibration(calibration_file, cals, geom, shape=shape, rpm=rpm, standard=standard,
                    sensitivity=sens, files=files, notes=notes, default_model=default_model)

# Read the file back as a check, one model at a time
print(f'saved {calibration_file}')
chk = {}
for model in cals:
    chk[model] = ss.load_calibration(calibration_file, model)
    print(f"  {model} model{' (default)' if model == default_model else ''}: "
          f"K0 = {chk[model]['K0']:.3g} m2/s, Kmin = {chk[model]['Kmin']:.3g} m2/s, {chk[model]['rpm']} rpm, "
          f"h = {chk[model]['geom'].h:.4f} m, sensor {chk[model]['geom'].z_obs:.4f} m above the bottom")

## 7. Optional: test the calibration on a second standard

The real test of the calibration: predict the decay of a standard of **different size or density** with each stored model, with no refitting. An rms near the noise level (about 0.04) means the turbulence model transfers; a clearly larger rms means something else (the assumed density or size distribution of one of the standards, or the 1-D model) is limiting.

In [ ]:
if validate:
    # Load the second standard's decay record
    dv = ss.load_decay(val_obs_file)

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
    ax[0].plot(dv['t'], dv['c'], 'k.', ms=3, label='second standard (data)')

    # Predict its decay with each saved model (no refitting)
    for model in cals:
        pred = ss.predict_standard(dv['t'], geom, chk[model], val_d_lo_um, val_d_hi_um, val_rho, T_C, S_ppt, coef,
                                   n_classes=n_classes)
        rms = float(np.sqrt(np.mean((pred - dv['c'])**2)))
        print(f"{model}: prediction of the second standard ({val_d_lo_um:.0f}-{val_d_hi_um:.0f} um, {val_rho:.0f} kg/m3): "
              f"rms = {rms:.4f}  (rms of the calibration fit: {cals[model]['rms']:.4f})")
        ax[0].plot(dv['t'], pred, '-', color=colors[model], label=f'{model} prediction, no refit')
        ax[1].plot(dv['t'], dv['c'] - pred, '-', color=colors[model], lw=0.8)

    ax[0].set_ylabel('c/c0')
    ax[0].legend()
    ax[1].axhline(0, color='gray')
    ax[1].set_xlabel('t since mixer off [s]')
    ax[1].set_ylabel('residual')
    plt.tight_layout()
else:
    print('validate = False: skipped')

## Notes

- A calibration applies to **one mixer rotation rate in one tank setup** (depth, sensor position). Step 3 warns if they differ.
- The ADV supplies the time *shape* only; $K_0$ cannot be derived from point ADV velocities without a length scale.
- Both models are stored; `default_model` is the one step 3 uses unless its `turb_model` input says otherwise. Without an ADV record the file holds only the exponential model.
- $K_{min}$ is far below what the ADV can resolve; treat it as an effective model parameter.
- A residual rms of about twice the sensor noise with lag-1 autocorrelation near 0.4 means something systematic remains; the standard's density or size distribution is the main suspect.